# OneBill Accounts & Contacts Sync

This Notebook uses API requests from OneBill and inserts the account and contact information into the SQL Server Management Studio database.

## Connect to OneBill and get Access Token

### OneBill Authentication

In [70]:
# pip install requests
import requests, pandas as pd

baseUrl = 'https://sandbox-sg.onebillsoftware.com'
subEndPoint = '/rest/SubscriberService/v1/subscribers'
accessTokenURL = f'{baseUrl}/oauth/token'

clientID = 'voyagersbx'
clientSecret = '13ad8f7edb3663a64cf1ce57a1860b68591669d2762a2de09c8dc78bdc03ed81'
username = 'api.usersbx'
password = '592bc33d78294f301df7b98752a4b7f0e6cdaab21555551f9a77377a0938976b'


### Getting Access Token

In [71]:
def get_AccessToken():
    tokenData = {
        'grant_type': 'password',
        'client_id': clientID,
        'client_secret': clientSecret,
        'username': username,
        'password': password
    }

    headers = {
        'Content-Type': 'application/x-www-form-urlencoded'
    }

    response = requests.post(accessTokenURL, data=tokenData, headers=headers)

    if response.status_code == 200:
        accessToken = response.json()
        return accessToken['access_token']
    else:
        print(f'Token Request Failed: {response.status_code} - {response.text}')
        return None

## Get Subscribers

In [72]:
def listSubscribers(access_token):
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }

    url = f'{baseUrl}{subEndPoint}'
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        subscribers = response.json().get('subscriber', [])
        return subscribers
    
    else:
        print(f'Failed to retrieve subscribers: {response.status_code} - {response.text}')
        return None

In [73]:
if __name__ == "__main__":
    token = get_AccessToken()
    subData = listSubscribers(token)

    print(subData)

[{'accountNumber': 'SR1002', 'accountName': 'TestAccount One', 'companyName': 'TestAccount One', 'createdDate': '2025-11-21T22:32:49+11:00', 'emailId': 'dilip.staging@gmail.com', 'phoneNumber': '', 'accountType': 1001, 'id': 2, 'sellerName': 'Internet Brands', 'accountStatus': 'Active', 'sellerPartyRoleId': '2', 'externalId': ''}, {'accountNumber': 'SR1202', 'accountName': 'Sam Smith', 'companyName': 'Sam Smith', 'createdDate': '2025-11-27T22:38:35+11:00', 'emailId': 'testcusta1@test.com', 'phoneNumber': '09121212122', 'accountType': 1001, 'id': 202, 'sellerName': 'Internet Brands', 'accountStatus': 'InCollections', 'sellerPartyRoleId': '2'}, {'accountNumber': 'SR1203', 'accountName': 'Navas T A', 'companyName': 'Navas T A', 'createdDate': '2025-11-27T23:35:49+11:00', 'emailId': 'navas@onebillsoftware.com', 'phoneNumber': '1234567890', 'accountType': 1001, 'id': 203, 'sellerName': 'Internet Brands', 'accountStatus': 'Active', 'sellerPartyRoleId': '2'}, {'accountNumber': 'SR1105', 'acco

### Convert API Request into Dataframe

In [74]:
subscribers_df = pd.DataFrame(subData)
print(subscribers_df.head())

  accountNumber      accountName      companyName                createdDate  \
0        SR1002  TestAccount One  TestAccount One  2025-11-21T22:32:49+11:00   
1        SR1202        Sam Smith        Sam Smith  2025-11-27T22:38:35+11:00   
2        SR1203        Navas T A        Navas T A  2025-11-27T23:35:49+11:00   
3        SR1105       Mark Smith       Mark Smith  2025-11-27T23:57:40+11:00   
4        SR1302      Mindy Smith      Mindy Smith  2025-11-28T03:38:21+11:00   

                         emailId  phoneNumber  accountType   id  \
0        dilip.staging@gmail.com                      1001    2   
1            testcusta1@test.com  09121212122         1001  202   
2      navas@onebillsoftware.com   1234567890         1001  203   
3  mark.mackay+sr1105@voyager.nz  04322323322         1001  104   
4       testuser0007@onebill.net   9018312312         1001  302   

        sellerName  accountStatus sellerPartyRoleId externalId  
0  Internet Brands         Active                 2

## Get Subscriber Details

In [75]:
def getSubscriberDetails(access_token, accountNumber):
    headers = {
        'Authorization': f'Bearer {access_token}',
        'Content-Type': 'application/json'
    }

    url = f'{baseUrl}{subEndPoint}/{accountNumber}'
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        details = response.json()
        return details
    else:
        print(f'Failed to retrieve details for {accountNumber}: {response.status_code} - {response.text}')
        return None

### Iterate Through Subscribers

Use the list of subscribers returned about to get and append all the details for each subscriber into a data frame

In [76]:
if __name__ == "__main__":
    # Assuming token is already obtained from previous cell
    # If not, uncomment: token = get_AccessToken()
    
    account_numbers = subscribers_df['accountNumber'].tolist()
    all_details = []
    
    for acc_num in account_numbers:
        details = getSubscriberDetails(token, acc_num)
        if details:
            all_details.append(details)
    
    sub_details_df = pd.DataFrame(all_details)
    sub_details_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 57 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   accountName                            13 non-null     object 
 1   accountNumber                          13 non-null     object 
 2   accountId                              13 non-null     object 
 3   id                                     13 non-null     object 
 4   createdDate                            13 non-null     object 
 5   accountStatus                          13 non-null     object 
 6   accountType                            13 non-null     object 
 7   primaryCurrencyId                      13 non-null     object 
 8   secondaryCurrencyId                    13 non-null     object 
 9   currencySymbol                         13 non-null     object 
 10  address                                9 non-null      object 
 11  contact 

In [77]:
sub_details_df[sub_details_df['accountingDisplayName'] == 'Mark Mackay']

,accountName,accountNumber,accountId,id,createdDate,accountStatus,accountType,primaryCurrencyId,secondaryCurrencyId,currencySymbol,...,isAllowOnlinePayment,isPreDelinquent,customerProfileRefKey,openTicketsCount,isOPGDown,status,isLeadToSubscriberConverted,leadConvertedDate,credit,annualRevenue
5,Mark Mackay,1403,403,403,2025-11-28T10:39:38+11:00,Active,1001,554,0,$,...,True,False,PGK62821856580890,2,False,OK,NaN,NaN,NaN,NaN


## Upsert Accounts


### Connect to SQL Server

In [ ]:
import sqlalchemy as sa
from sqlalchemy import create_engine

server = 'db02-qst.voyager.net.nz'  # or your server name
database = 'Sandbox'  # your database name
username = 'dom.woodhouse'  # if using SQL auth
password = 'Fifty Notion Begin8'  # if using SQL auth

# For Windows auth, use: connection_string = 'mssql+pyodbc:///?odbc_connect=DRIVER={ODBC Driver 17 for SQL Server};SERVER=localhost;DATABASE=OneBill;Trusted_Connection=yes;'

engine = create_engine(
    f'mssql+pyodbc://{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
)

### Create Account Table to Upsert
Clean up the Sub Details to only include details we need.
Currently: Account Name, Account Code, Created Date, Account Status, Account Type 

In [48]:


# Assuming sub_details_df is available from previous cell
# Select only the required columns
df_to_insert = sub_details_df[['accountingDisplayName', 'accountNumber', 'createdDate', 'accountStatus', 'accountType']]
df_to_insert = df_to_insert.rename(columns={
    'accountingDisplayName': 'AccountName',
    'accountNumber': 'AccountCode',
    'createdDate': 'CreatedDate',
    'accountStatus': 'AccountStatus',
    'accountType': 'AccountType'
})
# Convert createdDate to datetime if it's string
df_to_insert['CreatedDate'] = pd.to_datetime(df_to_insert['CreatedDate']).dt.tz_localize(None)

### Sync Accounts
When an Account exists, then replace the record with data from OneBill. If no Account exists then just append the account

In [49]:
from sqlalchemy import create_engine, text

# First, insert your data into a temporary staging table
df_to_insert.to_sql('Accounts_Staging', engine, schema='dbo', if_exists='replace', index=False)

# Then run a MERGE statement to upsert from staging into the real table
merge_query = text("""
    MERGE INTO dbo.Accounts AS target
    USING dbo.Accounts_Staging AS source
        ON target.AccountCode = source.AccountCode
    WHEN MATCHED THEN
        UPDATE SET
            target.AccountName   = source.AccountName,
            target.CreatedDate   = source.CreatedDate,
            target.AccountStatus = source.AccountStatus,
            target.AccountType   = source.AccountType
    WHEN NOT MATCHED BY TARGET THEN
        INSERT (AccountName, AccountCode, CreatedDate, AccountStatus, AccountType)
        VALUES (source.AccountName, source.AccountCode, source.CreatedDate, source.AccountStatus, source.AccountType);
""")

with engine.begin() as conn:
    conn.execute(merge_query)
    
    # Clean up the staging table
    conn.execute(text('DROP TABLE dbo.Accounts_Staging'))

print("Upsert completed successfully!")

   AccountCode
0       SR1002
1       SR1202
2       SR1203
3       SR1105
4       SR1302
5         1403
6         1603
7         1802
8         2002
9         2102
10   555544422
11        2103
12      SR1010


## Upsert Contacts

### Create Contacts Dataframe
Inside of the 'Subscriber Details' dataframe is the contacts details. Stored as a JSON Array, so expand these details and keep the account number

In [43]:
# Create contacts dataframe
contacts_expanded = sub_details_df[['accountNumber', 'contact']].explode('contact').reset_index(drop=True)
contacts = pd.concat([contacts_expanded.drop('contact', axis=1), pd.DataFrame(list(contacts_expanded['contact']))], axis=1)
contacts = contacts.rename(columns={'accountNumber': 'accountcode'})
print(contacts.tail())

   accountcode    id  contactType firstName lastName  \
10        2102  1304            0        Al      Ali   
11   555544422  1305            0   Boopath        y   
12        2103  1306            0   Shaneel      Pal   
13      SR1010  1405            0      Test  Contact   
14      SR1010  1404            0       Woo    House   

                                   communicationPoint  \
10  [{'type': 'CPHONE', 'value': ''}, {'type': 'AP...   
11  [{'type': 'CPHONE', 'value': '0223002412'}, {'...   
12  [{'type': 'EMAIL', 'value': 'shaneel.pal@voyag...   
13  [{'type': 'EMAIL', 'value': 'test@gmail.com'},...   
14  [{'type': 'CPHONE', 'value': ''}, {'type': 'PH...   

                                           userDetail  primaryContact  \
10  {'id': '1204', 'username': 'al-faris@ali.co.nz...            True   
11  {'id': '1205', 'username': 'boopathynz', 'user...            True   
12  {'id': '1206', 'username': 'shaneel.pal', 'use...            True   
13                          

### Expand Comm Details
This is again stored in JSON Array. So need to create columns for the 'type' and insert the 'values' into these columns.

In [45]:
import pandas as pd

# Explode the list to create one row per communication point
exploded = contacts.explode('communicationPoint')

# Extract 'type' and 'value' from the dicts
exploded['comm_type'] = exploded['communicationPoint'].apply(lambda x: x.get('type') if isinstance(x, dict) else None)
exploded['comm_value'] = exploded['communicationPoint'].apply(lambda x: x.get('value') if isinstance(x, dict) else None)

# Pivot to make 'type' into columns, with 'value' as the data
pivoted = exploded.pivot_table(index=exploded.index, columns='comm_type', values='comm_value', aggfunc='first')

# Rename columns if needed (e.g., prefix with 'comm_')
pivoted.columns = [f'comm_{col}' for col in pivoted.columns]

# Join back to the original DataFrame (drop duplicates if any)
contacts = contacts.join(pivoted, how='left')

# Drop the original column
contacts = contacts.drop('communicationPoint', axis=1)

In [46]:
contacts.tail()

,accountcode,id,contactType,firstName,lastName,userDetail,primaryContact,billingContact,locale,designation,registered,registrationLink,passwordSelectionMode,comm_APHONE,comm_CPHONE,comm_EMAIL,comm_PHONE
10,2102,1304,0,Al,Ali,"{'id': '1204', 'username': 'al-faris@ali.co.nz...",True,True,en_US,,True,NaN,0,,,al-faris@ali.co.nz,
11,555544422,1305,0,Boopath,y,"{'id': '1205', 'username': 'boopathynz', 'user...",True,True,en_US,,False,NaN,0,,0223002412,boopathy@voyager.nz,0223002412
12,2103,1306,0,Shaneel,Pal,"{'id': '1206', 'username': 'shaneel.pal', 'use...",True,True,en_US,,False,NaN,0,,0210727327,shaneel.pal@voyager.nz,
13,SR1010,1405,0,Test,Contact,NaN,False,True,en_US,,False,NaN,0,,6402056796,test@gmail.com,6402123456
14,SR1010,1404,0,Woo,House,NaN,True,False,en_US,,False,https://sandbox-sg.onebillsoftware.com/registr...,0,,,dom.woodhouse@voyager.nz,


In [51]:
contacts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15 entries, 0 to 14
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   accountcode            15 non-null     object
 1   id                     15 non-null     object
 2   contactType            15 non-null     int64 
 3   firstName              15 non-null     object
 4   lastName               15 non-null     object
 5   userDetail             12 non-null     object
 6   primaryContact         15 non-null     bool  
 7   billingContact         15 non-null     bool  
 8   locale                 15 non-null     object
 9   designation            15 non-null     object
 10  registered             15 non-null     bool  
 11  registrationLink       3 non-null      object
 12  passwordSelectionMode  15 non-null     int64 
 13  comm_APHONE            12 non-null     object
 14  comm_CPHONE            12 non-null     object
 15  comm_EMAIL             15

### Create Lean Contact DF

In [65]:
contacts_to_insert = contacts[[
    'accountcode', 'id', 'contactType', 'firstName', 'lastName', 'primaryContact', 'billingContact', 'comm_APHONE', 'comm_CPHONE', 'comm_EMAIL', 'comm_PHONE'
    ]]
contacts_to_insert = contacts_to_insert.rename(columns={
    'accountcode': 'AccountCode',
    'id': 'ContactID',
    'contactType': 'ContactType',
    'firstName': 'FirstName',
    'lastName': 'LastName',
    'primaryContact': 'PrimaryContact',
    'billingContact': 'BillingContact',
    'comm_APHONE': 'AlternatePhone',
    'comm_CPHONE': 'CellPhone',
    'comm_EMAIL': 'EmailAddress',
    'comm_PHONE': 'ContactPhone'
})
contacts_to_insert.head()

,AccountCode,ContactID,ContactType,FirstName,LastName,PrimaryContact,BillingContact,AlternatePhone,CellPhone,EmailAddress,ContactPhone
0,SR1002,5,0,TestAccount,One,True,True,,,dilip.staging@gmail.com,
1,SR1202,204,0,Sam,Smith,True,True,NaN,NaN,testcusta1@test.com,09121212122
2,SR1203,206,0,Navas,T A,True,True,NaN,NaN,navas@onebillsoftware.com,1234567890
3,SR1105,107,0,Mark,Smith,True,True,,,mark.mackay+sr1105@voyager.nz,04322323322
4,SR1302,404,0,Mindy,Smith,True,True,,,testuser0007@onebill.net,9018312312


## Sync Contacts
Use the same logic as the Accounts.

In [66]:
# Insert into staging table instead of directly into Contacts
contacts_to_insert.to_sql('Contacts_Staging', engine, schema='dbo', if_exists='replace', index=False)

merge_query_contacts = text("""
    MERGE INTO dbo.Contacts AS target
    USING dbo.Contacts_Staging AS source
        ON target.AccountCode = source.AccountCode  -- Assuming AccountCode is the key; adjust if needed
    WHEN MATCHED THEN
        UPDATE SET
            target.ContactID = source.ContactID,
            target.FirstName = source.FirstName,
            target.LastName = source.LastName,
            target.ContactType = source.ContactType,
            target.EmailAddress = source.EmailAddress,
            target.ContactPhone = source.ContactPhone,
            target.AlternatePhone = source.AlternatePhone,
            target.CellPhone = source.CellPhone
    WHEN NOT MATCHED BY TARGET THEN
        INSERT (AccountCode, ContactID, FirstName, LastName, ContactType, EmailAddress, ContactPhone, AlternatePhone, CellPhone)
        VALUES (source.AccountCode, source.ContactID, source.FirstName, source.LastName, source.ContactType, source.EmailAddress, source.ContactPhone, source.AlternatePhone, source.CellPhone);
""")

with engine.begin() as conn:
    conn.execute(merge_query_contacts)
    
    # Clean up the staging table
    conn.execute(text('DROP TABLE dbo.Contacts_Staging'))

print("Upsert completed successfully!")

Upsert completed successfully!
